# Flight Delay Prediction — Ingestion and Validation

Initial sample unit tests are examples only. Production 2025 checks and v1 publication appear below.

In [ ]:

# Flight Delay Prediction - Phase 1
# Project configuration

STORAGE_ACCOUNT = "stflight60106354"
YEAR = 2025
VERSION = "v1"

RAW_PATH = (
    f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"flight_delays/{YEAR}/{VERSION}/"
)

PROCESSED_PATH = (
    f"abfss://processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"flight_delays/{YEAR}/{VERSION}/"
)

CURATED_PATH = (
    f"abfss://curated@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
    f"flight_delays/{YEAR}/{VERSION}/"
)

TARGET = "ArrDel15"

print("Project: Flight Delay Prediction")
print("Dataset year:", YEAR)
print("Target:", TARGET)
print("Configuration ready.")


In [ ]:

from pyspark.sql import functions as F

def validate_flights(df):
    required_columns = [
        "Year", "Month", "DayofMonth",
        "Reporting_Airline", "Origin", "Dest",
        "CRSDepTime", "Cancelled", "Diverted",
        "ArrDelay"
    ]

    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    clean = df

    for col in ["Year", "Month", "DayofMonth"]:
        clean = clean.withColumn(col, F.col(col).cast("int"))

    for col in ["Cancelled", "Diverted", "ArrDelay"]:
        clean = clean.withColumn(col, F.col(col).cast("double"))

    clean = clean.withColumn(
        "ArrDel15",
        F.when(F.col("ArrDelay").isNotNull(),
               F.when(F.col("ArrDelay") >= 15, 1).otherwise(0))
    )

    invalid = (
        F.col("Year").isNull() |
        (F.col("Year") != 2025) |
        F.col("Month").isNull() |
        (~F.col("Month").between(1, 12)) |
        F.col("DayofMonth").isNull() |
        (~F.col("DayofMonth").between(1, 31)) |
        F.col("Reporting_Airline").isNull() |
        F.col("Origin").isNull() |
        F.col("Dest").isNull() |
        F.col("CRSDepTime").isNull() |
        F.col("Cancelled").isNull() |
        F.col("Diverted").isNull() |
        (F.col("Cancelled") != 0) |
        (F.col("Diverted") != 0) |
        F.col("ArrDelay").isNull()
    )

    accepted = clean.filter(~invalid)
    rejected = clean.filter(invalid)

    return accepted, rejected

print("Flight validation function created.")


In [ ]:

# Small example dataset for testing only
# This is NOT the real BTS dataset

sample_data = [
    (2025, 1, 10, "AA", "JFK", "LAX", 800, 0.0, 0.0, 20.0),
    (2025, 1, 11, "DL", "ATL", "ORD", 900, 0.0, 0.0, 5.0),
    (2025, 1, 12, "UA", "SFO", "DEN", 1000, 1.0, 0.0, None),
    (2025, 1, 13, "AA", "MIA", "JFK", 1100, 0.0, 0.0, None)
]

columns = [
    "Year", "Month", "DayofMonth",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "Cancelled", "Diverted",
    "ArrDelay"
]

df_test = spark.createDataFrame(sample_data, columns)

accepted, rejected = validate_flights(df_test)

print("Total rows:", df_test.count())
print("Accepted rows:", accepted.count())
print("Rejected rows:", rejected.count())

display(accepted)


In [ ]:

total = df_test.count()
good = accepted.count()
bad = rejected.count()

assert total == good + bad, "Row counts do not match!"
assert accepted.filter(F.col("ArrDel15").isNull()).count() == 0
assert good == 2
assert bad == 2

print("PASS: Row accounting is correct.")
print("PASS: Accepted records have valid targets.")
print("PASS: Example quality validation completed.")


In [ ]:

spark.sql("""
DESCRIBE EXTERNAL LOCATION loc_proj_flight60106354_raw
""").show(truncate=False)


In [ ]:
raw_path = "abfss://raw@stflight60106354.dfs.core.windows.net/flight_delays/2025/v1/"

files = dbutils.fs.ls(raw_path)

print("SUCCESS: Azure Storage connection works!")
print("Files found:", len(files))

for file in files:
    print(file.name, file.size)

assert len(files) == 13
print("PASS: 12 ZIP files and manifest found.")

In [ ]:

from pyspark.sql import functions as F

storage = "stflight60106354"

csv_path = (
    f"abfss://raw@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/csv/"
)

csv_files = [
    f for f in dbutils.fs.ls(csv_path)
    if f.name.endswith(".csv")
]

assert len(csv_files) == 12, "Expected 12 CSV files"

df_real = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(csv_path)
)

required = [
    "Year", "Month", "DayofMonth",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "Cancelled", "Diverted",
    "ArrDelay", "ArrDel15"
]

missing = [c for c in required if c not in df_real.columns]

assert not missing, f"Missing columns: {missing}"

print("PASS: Real 2025 CSV files loaded")
print("CSV files:", len(csv_files))
print("Columns:", len(df_real.columns))
print("Total flight records:", df_real.count())


In [ ]:

df_checked = (
    df_real
    .withColumn("YearNum", F.col("Year").cast("int"))
    .withColumn("MonthNum", F.col("Month").cast("int"))
    .withColumn("FlightDateParsed", F.to_date("FlightDate"))
    .withColumn("CancelledNum", F.col("Cancelled").cast("double"))
    .withColumn("DivertedNum", F.col("Diverted").cast("double"))
    .withColumn("ArrDelayNum", F.col("ArrDelay").cast("double"))
    .withColumn("TargetNum", F.col("ArrDel15").cast("double"))
    .withColumn("CRSDepNum", F.col("CRSDepTime").cast("int"))
)

valid_time = (
    (
        F.col("CRSDepNum").between(0, 2359)
        & ((F.col("CRSDepNum") % 100) < 60)
    )
    | (F.col("CRSDepNum") == 2400)
)

valid_condition = (
    (F.col("YearNum") == 2025)
    & F.col("MonthNum").between(1, 12)
    & F.col("FlightDateParsed").isNotNull()
    & (F.year("FlightDateParsed") == F.col("YearNum"))
    & (F.month("FlightDateParsed") == F.col("MonthNum"))
    & (F.dayofmonth("FlightDateParsed") == F.col("DayofMonth").cast("int"))
    & F.col("Reporting_Airline").isNotNull()
    & (F.trim("Reporting_Airline") != "")
    & F.col("Origin").isNotNull()
    & (F.trim("Origin") != "")
    & F.col("Dest").isNotNull()
    & (F.trim("Dest") != "")
    & valid_time
    & (F.col("CancelledNum") == 0)
    & (F.col("DivertedNum") == 0)
    & F.col("ArrDelayNum").isNotNull()
    & F.col("TargetNum").isin(0.0, 1.0)
    & (
        F.col("TargetNum")
        == F.when(F.col("ArrDelayNum") >= 15, 1.0).otherwise(0.0)
    )
)

df_checked = df_checked.withColumn(
    "QualityStatus",
    F.when(valid_condition, "ACCEPTED").otherwise("REJECTED")
)

quality_summary = (
    df_checked
    .groupBy("QualityStatus")
    .count()
)

display(quality_summary)

total = df_checked.count()
accepted_count = df_checked.filter(
    F.col("QualityStatus") == "ACCEPTED"
).count()
rejected_count = total - accepted_count

assert total == accepted_count + rejected_count

print("Total records:", total)
print("Accepted records:", accepted_count)
print("Rejected records:", rejected_count)
print("PASS: Row accounting verified")


In [ ]:

from pyspark.sql import functions as F

clean_flights = (
    df_checked
    .filter(F.col("QualityStatus") == "ACCEPTED")
    .select(
        F.col("FlightDateParsed").alias("FlightDate"),
        F.col("YearNum").alias("Year"),
        F.col("MonthNum").alias("Month"),
        F.trim("Reporting_Airline").alias("Reporting_Airline"),
        F.trim("Origin").alias("Origin"),
        F.trim("Dest").alias("Dest"),
        F.col("CRSDepNum").alias("CRSDepTime"),
        F.col("ArrDelayNum").alias("ArrDelay"),
        F.col("TargetNum").cast("int").alias("ArrDel15")
    )
    .withColumn(
        "DepHour",
        F.when(F.col("CRSDepTime") == 2400, 0)
         .otherwise(F.floor(F.col("CRSDepTime") / 100))
         .cast("int")
    )
    .withColumn("DayOfWeek", F.dayofweek("FlightDate"))
    .withColumn(
        "IsWeekend",
        F.when(F.col("DayOfWeek").isin(1, 7), 1).otherwise(0)
    )
    .withColumn("Quarter", F.quarter("FlightDate"))
    .withColumn("Route", F.concat_ws("-", "Origin", "Dest"))
)

assert clean_flights.count() == accepted_count
assert clean_flights.filter(F.col("ArrDel15").isNull()).count() == 0

print("PASS: Real cleaned dataset created")
print("Cleaned records:", accepted_count)
display(clean_flights.limit(10))


In [ ]:

processed_path = (
    "abfss://processed@stflight60106354.dfs.core.windows.net/"
    "flight_delays/2025/v1/clean_parquet/"
)

if accepted_count <= 0:
    raise ValueError("QUALITY FAILURE: No accepted flights")

acceptance_rate = accepted_count / total

if acceptance_rate < 0.90:
    raise ValueError("QUALITY FAILURE: Acceptance rate below 90%")

print(f"Acceptance rate: {acceptance_rate:.2%}")

(
    clean_flights
    .write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("Month")
    .save(processed_path)
)

print("SUCCESS: Real Parquet data published!")
print("Path:", processed_path)


In [ ]:

df_verify = spark.read.parquet(processed_path)

published_count = df_verify.count()

assert published_count == accepted_count

assert df_verify.filter(
    ~F.col("ArrDel15").isin(0, 1)
).count() == 0

print("PASS: Published Parquet row count matches")
print("Published records:", published_count)
print("PASS: Real data publication verified")


In [ ]:

from pyspark.sql import functions as F

# Inspect available flight-identification columns
candidate_keys = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

available = [c for c in candidate_keys if c in df_real.columns]

print("Available identification columns:", available)

if len(available) != len(candidate_keys):
    print("CHECK: Some identification columns are missing.")
else:
    print("PASS: All proposed identification columns available.")

# Check exact duplicate records across the full raw schema
exact_duplicate_rows = (
    df_real.groupBy(*df_real.columns)
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.sum(F.col("count") - 1).alias("duplicates")
    )
    .first()["duplicates"]
)

exact_duplicate_rows = int(exact_duplicate_rows or 0)

print("Exact duplicate rows:", exact_duplicate_rows)
print("PASS: Duplicate inspection completed.")


In [ ]:

from pyspark.sql import functions as F

flight_keys = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

assert all(c in df_real.columns for c in flight_keys)

duplicate_groups = (
    df_real.groupBy(*flight_keys)
    .count()
    .filter(F.col("count") > 1)
)

result = duplicate_groups.agg(
    F.count("*").alias("DuplicateGroups"),
    F.coalesce(
        F.sum(F.col("count") - 1), F.lit(0)
    ).alias("ExtraRows")
).first()

print("Duplicate flight-key groups:", result["DuplicateGroups"])
print("Extra rows sharing flight keys:", result["ExtraRows"])
print("PASS: Flight-key duplicate inspection completed.")

display(duplicate_groups.orderBy(F.desc("count")).limit(10))


In [ ]:

def enforce_quality_gate(total_rows, accepted_rows, minimum_rate=0.90):
    if total_rows <= 0:
        raise ValueError("QUALITY FAILURE: Empty dataset")

    rate = accepted_rows / total_rows
    print(f"Acceptance rate: {rate:.2%}")

    if rate < minimum_rate:
        raise ValueError("QUALITY FAILURE: Below 90% threshold")

    print("PASS: Quality gate accepted")
    return True

# Test a successful scenario
enforce_quality_gate(100, 98)

# Test a controlled failure
failure_detected = False

try:
    enforce_quality_gate(100, 60)
except ValueError as e:
    failure_detected = True
    print("EXPECTED FAILURE:", str(e))

assert failure_detected
print("PASS: Controlled quality-failure test completed.")
